# L7 - Datadog Experiments: Did the Optimization Actually Help?

## What You'll Learn

- How to turn real order data into a versioned Datadog **Dataset**
- How to run an **Experiment** that scores a prompt against that dataset
- How to compare a baseline prompt against the optimizer's candidate and get a number,
  rather than an opinion
- How **`LLMJudge`** grades answers with Bedrock, alongside a deterministic ground-truth check

## Overview

Notebook 1 drafted a revised system prompt and recorded it as bundle version 2. What it did
not do is establish whether version 2 is any better - it is a proposal, deliberately not
deployed.

This notebook makes that choice measurable. It builds a dataset from real orders, runs the
current prompt and the candidate prompt against it as two Datadog Experiments, and scores
both with the same evaluators so the comparison is like-for-like.

This is the offline counterpart to L6 notebook 4. There, evaluations graded live production
traffic as it arrived. Here, a fixed dataset is replayed against two configurations so the
only thing that changes is the prompt.

## Architecture

```
  DynamoDB CustomerOrders ---> Dataset (versioned, in Datadog)
                                  |
                    +-------------+-------------+
                    |                           |
          Experiment "prompt-v1"      Experiment "prompt-v2"
          config = current prompt     config = candidate prompt
                    |                           |
                    +-------------+-------------+
                                  |
                     Evaluators (identical for both)
                       - ground truth   (StringCheckEvaluator)
                       - answer quality (LLMJudge on Bedrock)
                       - pass rate      (summary evaluator)
                                  |
                     Compare in Datadog > Experiments
```

## Steps

1. Setup - enable the Agent Observability SDK and read config
2. Build the dataset from real orders
3. Define the task under test
4. Define the evaluators
5. Run the baseline experiment
6. Run the candidate experiment
7. Compare the two
8. Where this goes next


## License Details

Refer to LICENSE file in the main folder for license details.


## Prerequisites

- **L1-L3 completed** - `CustomerOrders` is populated and the Harness exists in SSM
- **L7 notebook 1 completed** - it writes the candidate prompt into the configuration bundle
  this notebook reads. If you skipped it, Step 6 shows how to supply a candidate by hand.
- Datadog credentials in SSM (L3 notebook 1, Step 6)

> This notebook is **read-only against AWS**. It reads orders from DynamoDB and prompts from
> the Harness and the configuration bundle. It never calls `update_harness`, so your deployed
> orchestrator is untouched.


## Step 1: Setup

Install the SDK and enable Agent Observability. The dataset and experiment APIs live in
`ddtrace.llmobs`, not in `datadog-api-client`, and they authenticate with the API key rather
than a scoped application key.


In [ ]:
%pip install --quiet ddtrace==4.12.2 boto3 pandas --upgrade

In [ ]:
import os, json
import boto3
from botocore.config import Config
from ddtrace.llmobs import LLMObs

REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
SSM_PREFIX = '/anycompany/agentcore'
ssm = boto3.client('ssm', region_name=REGION)

def get_ssm(name, default=None):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=True)['Parameter']['Value']
    except Exception:
        if default is not None:
            return default
        raise

DD_API_KEY = os.environ.get('DD_API_KEY') or get_ssm(f'{SSM_PREFIX}/dd_api_key')
DD_APP_KEY = os.environ.get('DD_APP_KEY') or get_ssm(f'{SSM_PREFIX}/dd_app_key')
DD_SITE    = os.environ.get('DD_SITE')    or get_ssm(f'{SSM_PREFIX}/dd_site', 'datadoghq.com')

ML_APP       = 'anycompany-customer-support'   # same app the agents report under
PROJECT_NAME = 'anycompany-agent-optimization'

# agentless_enabled: the notebook kernel has no local trace-agent to ship through.
LLMObs.enable(
    ml_app=ML_APP,
    project_name=PROJECT_NAME,
    api_key=DD_API_KEY,
    app_key=DD_APP_KEY,
    site=DD_SITE,
    agentless_enabled=True,
)

MODEL_ID = get_ssm(f'{SSM_PREFIX}/harness_model_id')
HARNESS_ID = get_ssm(f'{SSM_PREFIX}/harness_id')
HARNESS_ARN = get_ssm(f'{SSM_PREFIX}/harness_arn')
ORDERS_TABLE = get_ssm(f'{SSM_PREFIX}/dynamodb_orders_table')

bedrock = boto3.client('bedrock-runtime', region_name=REGION,
                       config=Config(read_timeout=120))
agentcore_control = boto3.client('bedrock-agentcore-control', region_name=REGION,
                                 config=Config(read_timeout=120))
ddb = boto3.client('dynamodb', region_name=REGION)

print('Datadog site :', DD_SITE)
print('ML app       :', ML_APP)
print('Project      :', PROJECT_NAME)
print('Judge model  :', MODEL_ID)
print('Orders table :', ORDERS_TABLE)

## Step 2: Build the Dataset from Real Orders

A dataset record has three parts: `input_data` (everything the model can see),
`expected_output` (ground truth), and `metadata` (for slicing results later).

The questions below are paired with the actual order rows from DynamoDB, so `expected_output`
is real data rather than something invented for the test. Passing the order context in
`input_data` is what makes this a *prompt* experiment: the model is not asked to fetch
anything, only to reason over what it is given, which is exactly what a system-prompt change
affects.

Datasets are versioned. Re-running this cell with changed records creates a new version
rather than overwriting the old one.


In [ ]:
CUSTOMER_ID = 'CUST-789'

# Pull this customer's real orders so ground truth comes from the data, not hand-authored.
resp = ddb.query(
    TableName=ORDERS_TABLE,
    KeyConditionExpression='customerId = :c',
    ExpressionAttributeValues={':c': {'S': CUSTOMER_ID}},
)

def flatten(item):
    """Minimal DynamoDB AttributeValue -> plain Python."""
    out = {}
    for k, v in item.items():
        if 'S' in v:
            out[k] = v['S']
        elif 'N' in v:
            out[k] = float(v['N'])
        elif 'BOOL' in v:
            out[k] = v['BOOL']
        elif 'M' in v:
            out[k] = flatten(v['M'])
        elif 'L' in v:
            out[k] = [flatten(x['M']) if 'M' in x else next(iter(x.values())) for x in v['L']]
    return out

orders = [flatten(i) for i in resp.get('Items', [])]
by_id = {o.get('orderId'): o for o in orders if o.get('orderId')}
print(f'Loaded {len(orders)} orders for {CUSTOMER_ID}')
for o in orders[:5]:
    print(f"  {o.get('orderId')}  {str(o.get('status')):12} {str(o.get('productName'))[:38]}")

def record(question, order_ids, expected, topic):
    """One dataset record: question + the order rows the model may reason over."""
    context = [by_id[oid] for oid in order_ids if oid in by_id]
    return {
        'input_data': {'question': question, 'orders': context},
        'expected_output': expected,
        'metadata': {'topic': topic, 'order_ids': list(order_ids)},
    }

# Ground truth is read straight off the rows so the dataset cannot drift from the data.
records = []
for oid in list(by_id)[:3]:
    o = by_id[oid]
    records.append(record(
        f'What is the status of order {oid}?', [oid], str(o.get('status', '')),
        'order_status'))
    records.append(record(
        f'What product did I order on {oid}?', [oid], str(o.get('productName', '')),
        'product_lookup'))

records.append(record(
    'How many orders do I have in total?',
    list(by_id), str(len(by_id)), 'aggregate'))

print(f'\nBuilt {len(records)} records')

dataset = LLMObs.create_dataset(
    dataset_name='anycompany-order-qa',
    project_name=PROJECT_NAME,
    description='Real CUST-789 orders with ground-truth answers, for prompt experiments.',
    records=records,
)
print(f'Dataset version {dataset.version} with {len(dataset)} records')
print('View:', dataset.url)

### Dataset in Datadog

**AI Observability > Improve > Datasets** lists the `anycompany-order-qa` dataset with its
records and version. Each row shows the input, the expected output (ground truth), and
any metadata - and every edit creates a new version, so a run always references a known
snapshot.

![Datadog Experiment Dataset](datadog_experiment_dataset.png)

## Step 3: The Task Under Test

An experiment calls `task(input_data, config)` once per record. `config` is set per
experiment, so putting the system prompt there is what makes the prompt the independent
variable: the same task function runs twice, and the only difference between the two runs is
`config['system_prompt']`.

The task calls Bedrock directly rather than invoking the Harness. That keeps the comparison
fast - a Converse call is a few seconds against roughly two minutes for a full Harness
invocation - and it isolates the prompt from tool selection and agent-to-agent delegation.
See Step 8 for the full-agent variant and what it costs.


In [ ]:
PROMPT_ID = 'anycompany-orchestrator-system-prompt'

def task(input_data, config):
    """Answer the question using only the supplied order rows."""
    question = input_data['question']
    orders = input_data.get('orders', [])

    user_text = (
        f"Customer orders (JSON):\n{json.dumps(orders, indent=2)}\n\n"
        f"Question: {question}"
    )

    # Prompt Tracking: attach the prompt's id/version/template to the auto-instrumented
    # Bedrock converse span so each call is tracked under Agent Observability > Prompts,
    # keyed by version. The experiment passes the version via config. (dd-trace v3.16.0+)
    prompt_meta = {
        'id': PROMPT_ID,
        'version': config.get('prompt_version', 'unversioned'),
        'chat_template': [
            {'role': 'system', 'content': config['system_prompt']},
            {'role': 'user', 'content': '{{user_text}}'},
        ],
        'variables': {'user_text': user_text},
    }

    with LLMObs.annotation_context(prompt=prompt_meta):
        resp = bedrock.converse(
            modelId=MODEL_ID,
            system=[{'text': config['system_prompt']}],
            messages=[{'role': 'user', 'content': [{'text': user_text}]}],
            inferenceConfig={'maxTokens': 512, 'temperature': 0.1},
        )
    return resp['output']['message']['content'][0]['text']


# Smoke-test the task before spending a full experiment run on it.
_probe = task(records[0]['input_data'], {'system_prompt': 'You are a helpful assistant.'})
print('Task returns:', _probe[:200].replace('\n', ' '), '...')

## Step 4: The Evaluators

Three, applied identically to both experiments:

| Evaluator | Type | What it measures |
|---|---|---|
| `StringCheckEvaluator(operation='icontains')` | built-in, deterministic | Does the answer contain the ground-truth value? |
| `LLMJudge(provider='bedrock')` | LLM-as-judge | Is the answer clear and complete, scored 1-10? |
| `GroundTruthPassRate` | summary | What fraction of records passed the ground-truth check? |

Deterministic and judge-based evaluators answer different questions. The string check is
objective but brittle - a correct answer phrased unusually can fail it. The judge handles
phrasing but is itself a model and can be inconsistent. Running both means a regression in
one is visible against the other.

`LLMJudge` supports Bedrock natively, so the judge uses the same model as the workshop with
no extra provider credentials.


In [ ]:
from ddtrace.llmobs import (LLMJudge, ScoreStructuredOutput,
                            BaseSummaryEvaluator, SummaryEvaluatorContext)
from ddtrace.llmobs.evaluators import StringCheckEvaluator

# 1. Deterministic: does the ground-truth value appear in the answer?
ground_truth = StringCheckEvaluator(
    operation='icontains',      # case-insensitive containment
    strip_whitespace=True,
    name='ground_truth_match',
)

# 2. LLM-as-judge on Bedrock, scored with an explicit pass threshold.
quality_judge = LLMJudge(
    provider='bedrock',
    model=MODEL_ID,
    name='answer_quality',
    system_prompt=(
        'You grade customer-support answers. Be strict about factual accuracy and '
        'reward answers that are complete and easy to read.'
    ),
    user_prompt=(
        'Question: {{input_data}}\n\n'
        'Answer given: {{output_data}}\n\n'
        'Ground truth: {{expected_output}}\n\n'
        'Score the answer from 1 to 10.'
    ),
    structured_output=ScoreStructuredOutput(
        description='Answer quality, 1 (poor) to 10 (excellent)',
        min_score=1,
        max_score=10,
        reasoning=True,
        min_threshold=7,          # 7 or above passes
    ),
)

# 3. Summary: aggregate the deterministic check across the whole dataset.
class GroundTruthPassRate(BaseSummaryEvaluator):
    """Fraction of records where the ground-truth check passed."""

    def __init__(self):
        super().__init__(name='ground_truth_pass_rate')

    def evaluate(self, context: SummaryEvaluatorContext):
        results = context.evaluation_results.get('ground_truth_match', [])
        if not results:
            return None
        passed = sum(1 for r in results if r is True or r == 'pass')
        return round(passed / len(results), 3)

pass_rate = GroundTruthPassRate()

EVALUATORS = [ground_truth, quality_judge]
SUMMARY_EVALUATORS = [pass_rate]
print('Evaluators ready:', [e.name for e in EVALUATORS], '+', [pass_rate.name])

## Step 5: Baseline - the Prompt Running in Production

The baseline is the system prompt the deployed Harness is using right now, read straight off
it. No modification, no redeploy.

`runs=2` replays the dataset twice. Agent output is non-deterministic, so a single pass can
make a prompt look better or worse than it is; averaging over two passes costs seconds here
and makes the comparison harder to fool. `jobs=4` evaluates records concurrently.


In [ ]:
harness = agentcore_control.get_harness(harnessId=HARNESS_ID)['harness']
BASELINE_PROMPT = '\n'.join(
    seg.get('text', '') for seg in harness.get('systemPrompt', []) if isinstance(seg, dict)
).strip()
if not BASELINE_PROMPT:
    raise RuntimeError('The Harness has no system prompt - run l3-orchestration/4 first.')

print(f'Baseline prompt: {len(BASELINE_PROMPT)} chars')
print(BASELINE_PROMPT[:300].replace('\n', ' '), '...\n')

exp_v1 = LLMObs.experiment(
    name='order-qa-prompt-v1-baseline',
    task=task,
    dataset=dataset,
    evaluators=EVALUATORS,
    summary_evaluators=SUMMARY_EVALUATORS,
    project_name=PROJECT_NAME,
    description='Deployed Harness system prompt, scored against the order-QA dataset.',
    tags={'prompt_version': 'v1', 'source': 'deployed-harness'},
    config={'system_prompt': BASELINE_PROMPT, 'prompt_version': 'v1'},
    runs=2,
)

result_v1 = exp_v1.run(jobs=4)
print('\nBaseline experiment complete.')
print('View:', exp_v1.url)

## Step 6: Candidate - the Revised Prompt

The candidate comes from the configuration bundle notebook 1 wrote
(`anycompany_harness_config`), whose latest version holds the revised prompt.

If the bundle is missing, or its latest version still matches the baseline because notebook 1's
draft step was skipped, set `CANDIDATE_PROMPT` by hand below and re-run. The experiment does
not care where the prompt came from - only that two prompts are being compared over one
dataset.


In [ ]:
CANDIDATE_PROMPT = ''          # paste a prompt here to override the bundle lookup

BUNDLE_NAME = 'anycompany_harness_config'

def _prompt_from_bundle():
    """Best-effort read of the recommended prompt out of the L7 nb1 bundle."""
    bundle_id = None
    for b in agentcore_control.list_configuration_bundles().get('bundles', []):
        if b.get('bundleName') == BUNDLE_NAME:
            bundle_id = b.get('bundleId')
            break
    if not bundle_id:
        return None, f"no bundle named '{BUNDLE_NAME}'"

    got = agentcore_control.get_configuration_bundle(bundleId=bundle_id)
    components = got.get('components') or {}
    # Components are keyed by the ARN of the resource they describe.
    entry = components.get(HARNESS_ARN) or (next(iter(components.values()), {}) or {})
    prompt = (entry.get('configuration') or {}).get('systemPrompt')
    if not prompt:
        return None, f'bundle {bundle_id} has no systemPrompt in its latest version'
    return prompt, f'bundle {bundle_id} version {got.get("versionId")}'

if not CANDIDATE_PROMPT:
    try:
        CANDIDATE_PROMPT, origin = _prompt_from_bundle()
        print('Candidate source:', origin)
    except Exception as e:
        CANDIDATE_PROMPT, origin = None, f'{type(e).__name__}: {e}'
        print('Bundle lookup failed:', origin)

if not CANDIDATE_PROMPT:
    raise RuntimeError(
        'No candidate prompt found. Run l7-agentic-operations/1_agent_optimization.ipynb '
        '(Steps 2 and 4) to produce one, or set CANDIDATE_PROMPT above.')

if CANDIDATE_PROMPT.strip() == BASELINE_PROMPT.strip():
    raise RuntimeError(
        'The candidate is identical to the baseline, so the experiment would compare a '
        'prompt against itself. Run notebook 1 Step 2 to draft a revision, or set '
        'CANDIDATE_PROMPT above.')

print(f'Candidate prompt: {len(CANDIDATE_PROMPT)} chars '
      f'({len(CANDIDATE_PROMPT) - len(BASELINE_PROMPT):+d} vs baseline)\n')

exp_v2 = LLMObs.experiment(
    name='order-qa-prompt-v2-candidate',
    task=task,
    dataset=dataset,                 # same dataset and version
    evaluators=EVALUATORS,           # same evaluators
    summary_evaluators=SUMMARY_EVALUATORS,
    project_name=PROJECT_NAME,
    description='Revised prompt from notebook 1, same dataset as the baseline.',
    tags={'prompt_version': 'v2', 'source': 'l7-nb1-candidate'},
    config={'system_prompt': CANDIDATE_PROMPT, 'prompt_version': 'v2'},
    runs=2,
)

result_v2 = exp_v2.run(jobs=4)
print('\nCandidate experiment complete.')
print('View:', exp_v2.url)

### Prompt Tracking: the two versions in Agent Observability

Because `task()` tagged each Bedrock call with the prompt's `id` and `version`, the v1 and
v2 runs above also populated **Agent Observability -> Prompts**. There you can compare the
two versions by call count, latency, tokens, and cost, view a text **diff** of the system
prompt, and jump to the exact spans each version produced - the observability complement to
the Experiment scores above. This pairs with **Prompt Management** (a central registry for
creating and versioning prompts) if you later register these prompts in Datadog.

In [ ]:
# The v1/v2 experiment runs above tagged their LLM spans with prompt id + version.
# Compare the two versions in Agent Observability > Prompts.
base = f'https://app.{DD_SITE}' if DD_SITE else 'https://app.datadoghq.com'
print('Prompt Tracking - compare prompt versions:')
print(f'  Open: {base}  ->  Agent Observability  ->  Prompts (left nav)')
print(f'  Filter:    @ml_app:{ML_APP}')
print(f'  Prompt id: {PROMPT_ID}   (versions: v1 baseline, v2 candidate)')

## Step 7: Compare

Both experiments ran the same dataset version through the same evaluators, so the scores are
directly comparable. The cell prints a local summary; the Datadog UI shows them side by side
with per-record drill-down.

Read it as a decision, not a verdict on the optimizer. A candidate that wins on the judge but
loses on the ground-truth check is usually more fluent and less accurate - which is worse for
a support agent, not better.


In [ ]:
import pandas as pd

def summarize(exp, label):
    df = exp.as_dataframe()
    row = {'experiment': label}
    for col in df.columns:
        name = col[-1] if isinstance(col, tuple) else col
        if 'ground_truth' in str(name) or 'answer_quality' in str(name):
            series = pd.to_numeric(df[col], errors='coerce')
            if series.notna().any():
                row[str(name)] = round(series.mean(), 3)
    return row, df

row_v1, df_v1 = summarize(exp_v1, 'v1 baseline')
row_v2, df_v2 = summarize(exp_v2, 'v2 candidate')

print('Mean scores across the dataset')
print('=' * 70)
print(pd.DataFrame([row_v1, row_v2]).to_string(index=False))

print('\nCompare side by side in Datadog:')
print('  v1:', exp_v1.url)
print('  v2:', exp_v2.url)
print('\nOr: AI Observability > Improve > Experiments, filtered to project '
      f"'{PROJECT_NAME}'.")
print('\nIf the columns above are empty, open the URLs - the UI is the source of truth for')
print('experiment scores, and the dataframe column layout varies by evaluator type.')

### Experiment Comparison in Datadog

**AI Observability > Improve > Experiments**, with the two runs selected, shows the baseline and
candidate side by side: the ground-truth pass rate and the judge score per experiment,
with per-record drill-down so you can see exactly which prompts changed the outcome.

![Datadog Experiment Comparison](datadog_experiment_comparison.png)

## Step 8: Where This Goes Next

Four extensions, in rough order of effort.

**Promote the winner.** If the candidate beat the baseline on both the ground-truth check and
the judge, apply bundle version 2 to the live Harness with `update_harness`. That is the only
mutating call in the L7 flow, so it is left for you to run deliberately rather than wired in
here.

**Test the full agent instead of the prompt alone.** Point `task` at `invoke_harness` rather
than Bedrock. That exercises registry discovery, tool selection, and delegation, but the
Harness prompt is fixed at deploy time, so comparing two prompts means calling
`update_harness` between experiments and reverting afterwards. Budget roughly two minutes per
record per experiment.

**Grow the dataset from production instead of authoring it.** In **AI Observability >
Investigate > Traces**, filter to the traces you care about (evaluation failures, latency outliers), click
**Automate Query**, set a sampling rate, and choose **Add to Dataset**. New matching traces are
routed in as they arrive, so the dataset tracks real behaviour. Automation-fed datasets cap at
20,000 records and are read-only - clone one to edit it. Single traces can also be added by
hand with **Add to Dataset**.

**Reuse a production evaluator offline.** `RemoteEvaluator(eval_name='...')` runs an
LLM-as-a-judge you configured in the Datadog UI as part of a local experiment - for example
the Language Mismatch evaluation from `l6-observability/4_datadog_quality_evaluations.ipynb`.
The reverse also exists: `LLMObs.publish_evaluator()` promotes a local `LLMJudge` to a managed
evaluation draft.

```python
from ddtrace.llmobs import RemoteEvaluator
lang = RemoteEvaluator(eval_name='Language Mismatch')
```

**Automate the loop.** Datadog's **Bits Evals** (Preview) does first-pass error analysis on
production failures, proposes dataset records and evaluators from what actually broke, and
compares baseline against candidate experiment results. It works through the Agent
Observability MCP Server, so the analysis runs from a coding agent with access to your traces.

## Summary

| What | How |
|---|---|
| Dataset | `LLMObs.create_dataset()` from real DynamoDB orders, auto-versioned |
| Task | Bedrock Converse with the prompt supplied via experiment `config` |
| Deterministic score | `StringCheckEvaluator(operation='icontains')` |
| Judge score | `LLMJudge(provider='bedrock')` with `ScoreStructuredOutput` |
| Aggregate | `BaseSummaryEvaluator` computing pass rate |
| Comparison | Two experiments, same dataset version and evaluators, differing only in `config` |

### L7 Complete

| Notebook | Role |
|----------|------|
| 1 - Agent Config Versioning | Drafts a revised prompt and records it as an immutable bundle version |
| 2 - Datadog Experiments | Measures whether that revision is actually an improvement |

AgentCore versions the configuration; Datadog decides what ships. AgentCore's span-based
evaluation, batch runs, and simulation need ADOT telemetry that this track disables, so those
live in the `aws-only` track.

## Cleanup

Experiments and datasets are Datadog-side objects and cost nothing to leave in place; the
dataset is useful to keep as a regression suite. The SDK has no dataset-delete call, so remove
them from **AI Observability > Improve > Datasets** and **> Experiments** if you want a clean slate.

Nothing in AWS was modified by this notebook, so there is nothing to revert.


In [ ]:
# === CLEANUP: nothing to tear down in AWS ===
# This notebook only read from DynamoDB, the Harness, and the configuration bundle.
#
# To drop the Datadog objects, delete them in the UI:
#   AI Observability > Improve > Datasets    -> anycompany-order-qa
#   AI Observability > Improve > Experiments -> order-qa-prompt-v1-baseline, order-qa-prompt-v2-candidate
#
# Flush any spans still buffered in this kernel:
# LLMObs.flush()
print('No AWS resources were created or modified by this notebook.')